# 07 — EDA: Year & Era Data

Explores the three year sources available for albums in the `valid_albums` scope:

| Column | Source |
|---|---|
| `release_group_meta_year` | `release_group_meta.first_release_date_year` — canonical MusicBrainz first-release year |
| `album_country_year` | `release_country.date_year` (earliest per album) — already in `mb_album_country.parquet` |
| `artist_begin_year` | `artist.begin_date_year` — band formation / artist birth year |

Covers: source coverage rates, agreement between sources, artist-year vs album-year displacement,
era distribution, outlier/data-quality review, and a sanity-check sample.

**Conclusion from this EDA:** `release_group_meta` + `release_unknown_country` overlap completely —
importing Source 1 alone closes 60% of unknowns. `release_alias` covers 1 album. Irreducible
unknown floor is ~2.1% of albums (no date anywhere in MusicBrainz).

**Inputs:** `data/mb_release_year.parquet`, `data/mb_album_country.parquet`,
`data/mb_album_artists.parquet`, `data/mb_artist.parquet`, `data/mb_album.parquet`

**Run after:** `01-postgres-to-parquet.ipynb` (to generate `mb_release_year.parquet`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '../data'

## Load & assemble

Replicates the join from `13-feature-era.ipynb` so this notebook is self-contained for review.
All three year sources are kept as separate columns.

In [ ]:
albums = pd.read_parquet(f'{DATA_DIR}/mb_album.parquet').rename(columns={'id': 'album_id'})

rg_year = pd.read_parquet(f'{DATA_DIR}/mb_release_year.parquet')

country_year = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_country.parquet', columns=['album_id', 'album_year'])
    .rename(columns={'album_year': 'album_country_year'})
)

album_artists = pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id', 'artist_name'])
artists = pd.read_parquet(f'{DATA_DIR}/mb_artist.parquet', columns=['id', 'artist_year']).rename(columns={'id': 'artist_id', 'artist_year': 'artist_begin_year'})
artist_year = album_artists.merge(artists, on='artist_id', how='left')[['album_id', 'artist_name', 'artist_begin_year']]

df = (
    albums[['album_id', 'name']]
    .merge(rg_year, on='album_id', how='left')
    .merge(country_year, on='album_id', how='left')
    .merge(artist_year, on='album_id', how='left')
)

print(f'Albums in scope: {len(df):,}')
df.head(3)

## Coverage per source

How many albums have a year from each source, and how many have at least one.

Three sources are available; they are checked independently before any fallback logic is applied:

- **`release_group_meta_year`** — the canonical answer from MusicBrainz. This is the earliest
  known release date across all individual releases of a release group (any country, any format),
  stored in `release_group_meta`. Imported as `mb_release_year.parquet`.
- **`album_country_year`** — the earliest country-specific date from `release_country`, already
  imported in `mb_album_country.parquet`. Semantically slightly different: it picks the earliest
  *country* release, which can differ from the global first release.
- **`artist_begin_year`** — band formation or artist birth year from `artist.begin_date_year`.
  A fallback of last resort: correct for era-binning purposes when no album date exists, but
  misleading for comeback albums released decades after formation.

The ~2.1% irreducible unknown floor has no date in any MusicBrainz source — confirmed by querying
`release_unknown_country` and `release_alias` as well (negligible additional coverage).

In [ ]:
n = len(df)
year_cols = ['release_group_meta_year', 'album_country_year', 'artist_begin_year']

print('Coverage per source:')
for col in year_cols:
    filled = df[col].notna().sum()
    print(f'  {col:<30} {filled:>8,}  ({filled/n*100:.1f}%)')

any_year = df[year_cols].notna().any(axis=1)
all_null  = (~any_year).sum()
print(f'\n  At least one year source       {any_year.sum():>8,}  ({any_year.mean()*100:.1f}%)')
print(f'  No year in any source          {all_null:>8,}  ({all_null/n*100:.1f}%)')

## Agreement between `release_group_meta_year` and `album_country_year`

For albums where both sources are present, how often do they agree?

`release_group_meta.first_release_date_year` is derived by MusicBrainz from the union of all
release events, so it should always be ≤ the earliest country date. A large gap (> 1 year)
typically means:

- A regional release happened years after the original (e.g. a US album released in Japan a
  decade later)
- A re-release or remaster was entered as a new release event but assigned to the original
  release group
- A data entry error in one of the two sources

Gaps of ≤ 1 year are expected (different regional release windows in the same year or adjacent
years). Gaps > 5 years are worth inspecting — the `nlargest` table at the bottom surfaces these.

In [ ]:
both = df[df['release_group_meta_year'].notna() & df['album_country_year'].notna()].copy()
both['year_diff'] = (both['release_group_meta_year'] - both['album_country_year']).abs()

print(f'Albums with both rg_meta and country year: {len(both):,}')
print(f'  Exact match (diff = 0) : {(both["year_diff"] == 0).sum():>8,}  ({(both["year_diff"] == 0).mean()*100:.1f}%)')
print(f'  Diff <= 1 year         : {(both["year_diff"] <= 1).sum():>8,}  ({(both["year_diff"] <= 1).mean()*100:.1f}%)')
print(f'  Diff <= 5 years        : {(both["year_diff"] <= 5).sum():>8,}  ({(both["year_diff"] <= 5).mean()*100:.1f}%)')
print(f'  Diff > 5 years         : {(both["year_diff"] > 5).sum():>8,}  ({(both["year_diff"] > 5).mean()*100:.1f}%)')
print(f'\nLargest gaps:')
print(both.nlargest(10, 'year_diff')[['album_id', 'name', 'artist_name', 'release_group_meta_year', 'album_country_year', 'year_diff']].to_string(index=False))

## Artist formation year vs album release year

For albums where both `release_group_meta_year` and `artist_begin_year` are available,
how far apart are they?

This matters because `artist_begin_year` is the **fallback of last resort** in the Option D
chain. It is only used when neither album-level source has a year, so the question is: how
misleading is it when it does fire?

- A gap of 0–10 years is fine — the album was released early in the artist's career.
- A gap of 10–20 years is acceptable — mid-career.
- A gap > 20 years is the **comeback-album problem**: an artist formed in 1965 releasing an
  album in 2005 would be binned to the 1960s based on formation year, when the album is
  genuinely a 2000s record.

This distribution tells us how much noise `artist_begin_year` introduces as a fallback.
If the median gap is large, it is worth tracking `best_year_source == 'artist_begin'` albums
separately and either capping the fallback or marking them as lower-confidence era assignments.

In [ ]:
both_artist = df[df['release_group_meta_year'].notna() & df['artist_begin_year'].notna()].copy()
both_artist['artist_album_gap'] = both_artist['release_group_meta_year'] - both_artist['artist_begin_year']

print(f'Albums with both rg_meta year and artist begin year: {len(both_artist):,}')
print(f'\nGap (album_year - artist_begin_year) distribution:')
print(both_artist['artist_album_gap'].describe().round(1))
print(f'\nAlbums released before artist formed (negative gap): {(both_artist["artist_album_gap"] < 0).sum():,}')
print(f'Albums released 20+ years after artist formed      : {(both_artist["artist_album_gap"] >= 20).sum():,}')

## Era distribution

Albums per era bin after the Option D fallback chain (`rg_meta → country → artist`).

**Binning logic:**
- Years before 1900 and 1900–1949 use **50-year buckets** — there are too few albums in these
  ranges for decade bins to be meaningful; grouping them keeps each bucket dense enough to
  produce useful similarity signal.
- 1950 onward uses **decade bins** (1950s, 1960s, …) — the data is dense enough here.
- Years > 2026 are hard-capped to `Unknown` regardless of source — they are either
  MusicBrainz data-entry errors or genuine future-dated pre-releases, neither of which should
  influence era matching.

The `YEAR_CORRECTIONS` dict patches the five confirmed bad years found during outlier review
(see the outliers cell below) before binning is applied.

The grey bars in the chart (`Pre-1900`, `1900–1949`, `Unknown`) are the sparse/invalid
categories. The blue bars from `1950s` onward should form a roughly bell-shaped distribution
peaking somewhere in the 1990s–2000s for this album scope.

In [ ]:
# Reproduce the Option D chain and era bins (mirrors feature-era notebook)
YEAR_CORRECTIONS = {
    4576816: 2021,
    4598782: 1969,
    4615228: 2025,
    4643359: 2022,
    4706113: 2026,
}

df['best_year'] = (
    df['release_group_meta_year']
    .combine_first(df['album_country_year'])
    .combine_first(df['artist_begin_year'])
)
for album_id, yr in YEAR_CORRECTIONS.items():
    df.loc[df['album_id'] == album_id, 'best_year'] = float(yr)

def assign_era(year):
    if pd.isna(year): return 'Unknown'
    y = int(year)
    if y > 2026: return 'Unknown'
    if y < 1900: return 'Pre-1900'
    if y < 1950: return '1900–1949'
    return f'{(y // 10) * 10}s'

df['era_bin'] = df['best_year'].apply(assign_era)

era_order = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)] + ['Unknown']
era_counts = df['era_bin'].value_counts().reindex(era_order, fill_value=0)

print('Albums per era bin:')
print(era_counts.to_string())

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#aaa' if e in ('Pre-1900', '1900–1949', 'Unknown') else '#4C72B0' for e in era_counts.index]
ax.bar(era_counts.index, era_counts.values, color=colors)
ax.set_xlabel('Era')
ax.set_ylabel('Albums')
ax.set_title('Album count by era (Option D: rg_meta → country → artist fallback)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Outlier / data quality check

Albums with `best_year < 1900` or `best_year > 2026` before corrections and capping.

MusicBrainz is crowdsourced, so year typos are common. The most frequent patterns found:
- **Digit transposition**: `2205` → `2025`, `2202` → `2022` (swap last two digits)
- **Millennium flip**: `2069` → `1969` (leading `2` instead of `1`)
- **Off-by-one**: `2027` entered for a 2026 release (not yet committed to DB at time of entry)

Each outlier was verified via web search. Confirmed corrections are stored in `YEAR_CORRECTIONS`
in `13-feature-era.ipynb` and mirrored in the `dist` cell above for this self-contained EDA view.

The raw (pre-correction) best_year is used here so the original bad values are visible for audit.
After corrections, `assign_era()` in `13-feature-era.ipynb` hard-caps any remaining `> 2026`
values to `Unknown` rather than trying to infer the correct year.

In [ ]:
# Recompute raw best_year (before corrections) to show original bad values
raw_best = (
    df['release_group_meta_year']
    .combine_first(df['album_country_year'])
    .combine_first(df['artist_begin_year'])
)

pre1900  = df[raw_best < 1900].copy()
post2026 = df[raw_best > 2026].copy()
pre1900['raw_year']  = raw_best[pre1900.index]
post2026['raw_year'] = raw_best[post2026.index]

print(f'Albums with raw year < 1900 : {len(pre1900):,}')
if len(pre1900):
    print(pre1900[['album_id', 'name', 'artist_name', 'raw_year']].to_string(index=False))

print(f'\nAlbums with raw year > 2026 : {len(post2026):,}')
if len(post2026):
    print(post2026[['album_id', 'name', 'artist_name', 'raw_year']].to_string(index=False))

## Sanity check — sample of albums with all three sources

Spot-check albums where all three year columns are populated to confirm values are plausible
and that the fallback chain would have selected the right source.

What to look for:
- `release_group_meta_year` and `album_country_year` should be equal or within 1–2 years
- `artist_begin_year` should be ≤ `release_group_meta_year` (an artist can't release before forming)
- `era_bin` should match the decade you'd expect for a well-known album
- `best_year` should equal `release_group_meta_year` for all rows here (it is Source 1, so it
  wins the fallback chain whenever it is non-null)

In [ ]:
three_sources = df[
    df['release_group_meta_year'].notna() &
    df['album_country_year'].notna() &
    df['artist_begin_year'].notna()
].sample(15, random_state=42)

three_sources[[
    'album_id', 'name', 'artist_name',
    'release_group_meta_year', 'album_country_year', 'artist_begin_year',
    'best_year', 'era_bin'
]]